# Análisis univariado

In [1]:
from eda_utils import *   # carga y prepara la base (capítulo 1)

In [2]:
p = viv.precio_M
fig = make_subplots(1, 2, subplot_titles=("Precio (hasta el percentil 99)", "Precio en escala logarítmica"))
fig.add_histogram(x=p[p <= p.quantile(0.99)], nbinsx=60, marker_color=AZUL, row=1, col=1,
                  hovertemplate="%{x} M: %{y} ventas<extra></extra>")
fig.add_vline(x=p.median(), line_color=ROJO, line_width=2, row=1, col=1,
              annotation_text=f"mediana {p.median():,.0f} M")
fig.add_histogram(x=np.log10(viv.precio_real), nbinsx=60, marker_color=VERDE, row=1, col=2,
                  hovertemplate="%{y} ventas<extra></extra>")
marcas = [1e7, 3e7, 1e8, 3e8, 1e9, 3e9]
fig.update_xaxes(tickvals=np.log10(marcas), ticktext=["10 M", "30 M", "100 M", "300 M", "1.000 M", "3.000 M"], row=1, col=2)
fig.update_xaxes(title_text="millones de pesos de 2025", row=1, col=1)
fig.update_layout(showlegend=False, bargap=0.03)
mostrar(fig, "Distribución del precio de venta")

viv[["precio_real", "area", "precio_m2_real"]].describe(percentiles=[.05, .25, .5, .75, .95]).T

,count,mean,std,min,5%,25%,50%,75%,95%,max
precio_real,"15,459.00","293,112,804.63","354,492,301.37","14,944,446.34","58,293,444.34","131,185,338.84","209,202,742.67","348,562,189.57","792,037,996.53","23,116,325,157.17"
area,"15,459.00",102.28,428.86,5.00,40.00,51.00,70.00,113.00,248.00,"52,168.00"
precio_m2_real,"15,459.00","3,294,231.43","2,232,009.30","310,661.49","678,959.62","1,793,489.00","2,993,827.81","4,430,215.84","6,616,033.74","40,000,675.46"


La vivienda mediana cuesta 209 M de 2025 y tiene 70 m². El precio tiene una cola
larga a la derecha que desaparece en escala logarítmica.

In [3]:
fig = make_subplots(1, 2, subplot_titles=("Precio", "log10(precio)"))
for i, col in enumerate(["precio_real", "log_precio"], 1):
    (teo, obs), (pend, corte, _) = stats.probplot(viv[col], dist="norm")
    fig.add_scatter(x=teo, y=obs, mode="markers", marker=dict(size=3, color=AZUL, opacity=0.4),
                    row=1, col=i, hoverinfo="skip")
    fig.add_scatter(x=teo, y=pend * teo + corte, mode="lines", line=dict(color=ROJO), row=1, col=i)
fig.update_xaxes(title_text="cuantiles de una normal")
fig.update_layout(showlegend=False)
mostrar(fig, "Gráficos Q-Q: qué tan cerca de la normal está el precio", alto=400)

En pesos el precio se aleja por completo de la normal; en logaritmo sigue la recta salvo
en los extremos (asimetría de 0,14). Por eso el modelo se entrenará sobre log(precio).

In [4]:
fig = make_subplots(1, 4, subplot_titles=("Habitaciones", "Baños", "Piso", "Edad (años)"))
for i, (col, tope) in enumerate([("habitaciones", 7), ("banios", 6), ("piso", 20)], 1):
    vc = viv[col].dropna().round().astype(int).clip(upper=tope).value_counts().sort_index()
    etiquetas = [str(v) if v < tope else f"{tope}+" for v in vc.index]
    fig.add_bar(x=etiquetas, y=vc.values, marker_color=AZUL, row=1, col=i,
                hovertemplate="%{x}: %{y} ventas<extra></extra>")
fig.add_histogram(x=viv.edad, nbinsx=40, marker_color=VERDE, row=1, col=4,
                  hovertemplate="%{x} años: %{y} ventas<extra></extra>")
fig.update_layout(showlegend=False, bargap=0.1)
mostrar(fig, "Características de las viviendas", alto=380)

viv[["habitaciones", "banios", "piso", "anio_construccion", "edad"]].describe(percentiles=[.05, .5, .95]).T

,count,mean,std,min,5%,50%,95%,max
habitaciones,"15,452.00",2.85,0.88,0.00,2.00,3.00,4.00,14.00
banios,"15,458.00",1.71,0.84,0.00,1.00,2.00,3.00,20.00
piso,"15,459.00",3.70,3.30,1.00,1.00,2.50,10.00,30.00
anio_construccion,"15,454.00","2,007.86",15.11,"1,940.00","1,986.00","2,014.00","2,022.00","2,025.00"
edad,"14,825.00",14.90,15.37,0.00,1.00,9.00,39.00,85.00


La vivienda típica tiene 3 habitaciones, 2 baños y 9 años de antigüedad. La mayoría
está en pisos bajos.

In [5]:
fig = make_subplots(1, 2, subplot_titles=("Estrato", "Tipo de predio"), horizontal_spacing=0.15)
e = viv.estrato_txt.value_counts().reindex(NOMBRES_ESTRATO + ["Sin estrato"]).dropna()
fig.add_bar(x=e.index.str.replace("Estrato ", ""), y=e.values, row=1, col=1,
            marker_color=COLORES_ESTRATO + ["#BBBBBB"], hovertemplate="%{x}: %{y} ventas<extra></extra>")
c = viv.condicion_predio.value_counts().sort_values()
fig.add_bar(x=c.values, y=c.index, orientation="h", row=1, col=2, marker_color=ROJO,
            hovertemplate="%{y}: %{x} ventas<extra></extra>")
fig.update_layout(showlegend=False)
mostrar(fig, "Estrato y tipo de predio", alto=380)

Los estratos 1 a 4 suman el 82% de las ventas y el 60% son unidades en propiedad
horizontal, es decir, apartamentos.